# Análise complementar controlada do número de objetivos

Este notebook acrescenta análises controladas ao benchmark principal, sem reexecutar qualquer método de otimização. Para cada família (`low`, `medium` e `high`), parte-se exclusivamente das 12 funções do cenário correspondente: o problema com quatro objetivos usa as quatro primeiras funções, o problema com seis preserva essas quatro e acrescenta duas, e o problema com 12 usa todas. Assim, dentro de cada família, as funções comuns são literalmente idênticas e apenas novos objetivos são acrescentados.

Os prefixos são apresentados mesmo quando a correlação resultante não atende à tolerância originalmente usada na calibração, sempre com o valor observado explícito. A figura adicional entre `m12_low`, `m12_medium` e `m12_high` permanece apenas descritiva: como suas âncoras foram calibradas separadamente, ela representa geometrias associadas a diferentes níveis de correlação, mas não identifica isoladamente um efeito causal da correlação. Todas as pranchas usam a projeção direta $f_1\times f_2$, na escala original, sem componentes principais, variáveis latentes ou normalização.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.spatial import Delaunay

def project_root(start=Path.cwd()):
    path = start.resolve()
    for candidate in (path, *path.parents):
        if (candidate / 'configs' / 'full.json').exists():
            return candidate
    raise FileNotFoundError('Raiz do projeto não encontrada.')

ROOT = project_root()
CFG = json.loads((ROOT / 'configs' / 'full.json').read_text(encoding='utf-8'))
DIAGNOSTICS = pd.read_csv(ROOT / 'data' / 'generated' / 'scenario_diagnostics.csv').set_index('scenario')
OUT_DIR = ROOT / 'results' / 'synthetic' / 'figures' / 'true_pareto_comparisons'
OUT_DIR.mkdir(parents=True, exist_ok=True)

OBJECTIVE_PAIR = (0, 1)
DISPLAY_POINTS = 8000
PLOT_SEED = 20260825
POINT_COLOR = '#31688e'
ANCHOR_COLOR = 'crimson'

FIGURE_1 = [
    (4, r'$m=4$'),
    (6, r'$m=6$'),
    (12, r'$m=12$'),
]
CONTROLLED_FAMILIES = {'low': 'baixa', 'medium': 'média', 'high': 'alta'}
FIGURE_2 = [
    ('m12_low', 'correlação baixa'),
    ('m12_medium', 'correlação média'),
    ('m12_high', 'correlação alta'),
]

plt.rcParams.update({
    'font.family': 'DejaVu Serif',
    'font.size': 13,
    'axes.titlesize': 16,
    'axes.labelsize': 16,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 14,
    'savefig.facecolor': 'white',
    'axes.facecolor': 'white',
})

print('Raiz:', ROOT)
print('Saída:', OUT_DIR)

In [ ]:
def _scenario_seed(scenario):
    return PLOT_SEED + sum((index + 1) * ord(char) for index, char in enumerate(scenario))

def truth(X, anchors):
    return np.sum((np.asarray(X)[:, None, :] - anchors[None, :, :]) ** 2, axis=2)

def sample_convex_hull(anchors, n, seed):
    triangulation = Delaunay(anchors)
    tetrahedra = anchors[triangulation.simplices]
    volumes = np.abs(np.linalg.det(tetrahedra[:, 1:] - tetrahedra[:, :1])) / 6
    rng = np.random.default_rng(seed)
    remaining = n - len(anchors)
    tetrahedron_ids = rng.choice(len(tetrahedra), size=remaining, p=volumes / volumes.sum())
    exponential = rng.exponential(size=(remaining, 4))
    weights = exponential / exponential.sum(axis=1, keepdims=True)
    interior = np.einsum('ni,nij->nj', weights, tetrahedra[tetrahedron_ids])
    return np.vstack([anchors, interior])

def display_subset(F, key):
    if len(F) <= DISPLAY_POINTS:
        return F
    rng = np.random.default_rng(_scenario_seed(str(key)))
    indices = np.sort(rng.choice(len(F), size=DISPLAY_POINTS, replace=False))
    return F[indices]

def load_benchmark_projection(scenario):
    reference_path = ROOT / 'data' / 'reference_fronts' / f'{scenario}_pareto_reference.npz'
    scenario_path = ROOT / 'data' / 'generated' / f'{scenario}_scenario.npz'
    if not reference_path.exists() or not scenario_path.exists():
        raise FileNotFoundError(f'Artefatos ausentes para {scenario}.')

    with np.load(reference_path, allow_pickle=False) as reference:
        F = np.asarray(reference['F'], dtype=float)
    with np.load(scenario_path, allow_pickle=False) as scenario_data:
        anchors = np.asarray(scenario_data['anchors'], dtype=float)

    expected = int(CFG['reference_points'])
    assert len(F) == expected, f'{scenario}: referência com {len(F)} pontos; esperado {expected}.'
    Fanchors = np.sum((anchors[:, None, :] - anchors[None, :, :]) ** 2, axis=2)
    assert np.isfinite(F).all() and np.isfinite(Fanchors).all()
    assert F.shape[1] >= 2

    F_display = display_subset(F, scenario)

    achieved = float(DIAGNOSTICS.loc[scenario, 'achieved'])
    target = float(DIAGNOSTICS.loc[scenario, 'target'])
    return F_display[:, OBJECTIVE_PAIR], Fanchors[:, OBJECTIVE_PAIR], {
        'scenario': scenario,
        'm': int(F.shape[1]),
        'target_correlation': target,
        'achieved_correlation': achieved,
        'reference_points': int(len(F)),
        'display_points': int(len(F_display)),
        'design': 'representative_benchmark',
        'source_scenario': scenario,
    }

def load_controlled_family_projection(m, level):
    source_scenario = f'm12_{level}'
    scenario_path = ROOT / 'data' / 'generated' / f'{source_scenario}_scenario.npz'
    reference_path = ROOT / 'data' / 'reference_fronts' / f'{source_scenario}_pareto_reference.npz'
    with np.load(scenario_path, allow_pickle=False) as scenario_data:
        anchors12 = np.asarray(scenario_data['anchors'], dtype=float)
        correlation12 = np.asarray(scenario_data['correlation'], dtype=float)
    anchors = anchors12[:m].copy()
    assert np.array_equal(anchors, anchors12[:m])
    if m == 12:
        with np.load(reference_path, allow_pickle=False) as reference:
            F = np.asarray(reference['F'], dtype=float)
    else:
        X = sample_convex_hull(anchors, int(CFG['reference_points']), 20260825 + m)
        F = truth(X, anchors)
    Fanchors = truth(anchors, anchors)
    correlation = correlation12[:m, :m]
    achieved = float(np.mean(np.abs(correlation[np.triu_indices(m, 1)])))
    target = float(DIAGNOSTICS.loc[source_scenario, 'target'])
    tolerance = float(CFG['correlation_tolerance'])
    F_display = display_subset(F, f'controlled-{level}-m{m}')
    assert len(F) == int(CFG['reference_points']) and np.isfinite(F).all()
    return F_display[:, OBJECTIVE_PAIR], Fanchors[:, OBJECTIVE_PAIR], {
        'scenario': f'controlled_m{m}_{level}',
        'm': m,
        'family': level,
        'target_correlation': target,
        'achieved_correlation': achieved,
        'correlation_deviation': achieved - target,
        'within_original_tolerance': bool(abs(achieved - target) <= tolerance),
        'reference_points': int(len(F)),
        'display_points': int(len(F_display)),
        'design': 'controlled_nested_family_prefix',
        'source_scenario': source_scenario,
    }

def shared_limits(loaded):
    all_points = np.vstack([array for _, _, F, Fanchors, _ in loaded for array in (F, Fanchors)])
    lower = np.min(all_points, axis=0); upper = np.max(all_points, axis=0)
    span = np.maximum(upper - lower, 1e-12); padding = 0.04 * span
    return (lower[0] - padding[0], upper[0] + padding[0]), (lower[1] - padding[1], upper[1] + padding[1])

def style_objective_axis(ax, xlim, ylim):
    ax.set_xlim(*xlim); ax.set_ylim(*ylim)
    ax.set_xlabel(r'$f_1$'); ax.set_ylabel(r'$f_2$')
    ax.set_aspect('equal', adjustable='box')
    ax.grid(alpha=0.16, linewidth=0.55)

def make_comparison_figure(specification, loader, stem):
    loaded = [(key, label, *loader(key)) for key, label in specification]
    xlim, ylim = shared_limits(loaded)
    fig, axes = plt.subplots(1, 3, figsize=(16.8, 5.6), layout='constrained')
    manifest = []
    letters = 'abc'
    for position, (ax, (scenario, label, F, Fanchors, info)) in enumerate(zip(axes, loaded), start=1):
        ax.scatter(
            F[:, 0], F[:, 1], s=4.0, c=POINT_COLOR, alpha=0.22,
            linewidths=0, rasterized=True,
        )
        ax.scatter(
            Fanchors[:, 0], Fanchors[:, 1], marker='*', s=140,
            c=ANCHOR_COLOR, edgecolors='.15', linewidths=0.55, zorder=5,
        )
        style_objective_axis(ax, xlim, ylim)
        ax.set_title(
            f'({letters[position - 1]}) {label}\n'
            + rf'$\bar{{\rho}}={info["achieved_correlation"]:.3f}$',
            pad=7,
        )
        manifest.append({'figure': stem, **info, 'projection': 'f1-f2', 'normalization': 'none'})

    legend = [
        Line2D([0], [0], marker='o', linestyle='none', markerfacecolor=POINT_COLOR,
               markeredgecolor='none', alpha=0.55, markersize=6,
               label='fronteira verdadeira (amostra para exibição)'),
        Line2D([0], [0], marker='*', linestyle='none', markerfacecolor=ANCHOR_COLOR,
               markeredgecolor='white', markeredgewidth=0.55, markersize=10,
               label='ótimos individuais verdadeiros'),
    ]
    fig.legend(handles=legend, loc='outside lower center', ncol=2, frameon=False)
    png = OUT_DIR / f'{stem}.png'
    pdf = OUT_DIR / f'{stem}.pdf'
    fig.savefig(png, dpi=300, bbox_inches='tight')
    fig.savefig(pdf, dpi=300, bbox_inches='tight')
    plt.close(fig)
    return png, pdf, manifest

In [ ]:
controlled_stems = {
    'low': 'figura_1a_familia_low_prefixos_m_f1_f2',
    'medium': 'figura_1b_familia_medium_prefixos_m_f1_f2',
    'high': 'figura_1c_familia_high_prefixos_m_f1_f2',
}
controlled_artifacts = []
controlled_manifests = []
for level in CONTROLLED_FAMILIES:
    def loader(m, selected_level=level):
        return load_controlled_family_projection(m, selected_level)
    png, pdf, rows = make_comparison_figure(FIGURE_1, loader, controlled_stems[level])
    controlled_artifacts.extend([png, pdf])
    controlled_manifests.extend(rows)

png_2, pdf_2, manifest_2 = make_comparison_figure(
    FIGURE_2,
    load_benchmark_projection,
    'figura_2_geometrias_representativas_correlacao_f1_f2',
)

manifest = pd.DataFrame(controlled_manifests + manifest_2)
manifest['png'] = manifest['figure'].map(lambda stem: (OUT_DIR / f'{stem}.png').relative_to(ROOT).as_posix())
manifest['pdf'] = manifest['figure'].map(lambda stem: (OUT_DIR / f'{stem}.pdf').relative_to(ROOT).as_posix())
manifest.to_csv(OUT_DIR / 'true_pareto_comparison_manifest.csv', index=False)

metadata = {
    'source': '100000-point true Pareto reference generated directly from the convex hull of the anchors',
    'normalization': 'none; original objective-response scale',
    'projection': ['f1', 'f2'],
    'display_sample_points_per_panel': DISPLAY_POINTS,
    'display_sample_seed': PLOT_SEED,
    'view': 'direct two-dimensional objective projection with shared limits within each figure',
    'controlled_family_design': 'within each family, nested prefixes of its existing m12 functions; only new objectives are added',
    'controlled_objective_prefixes': [m for m, _ in FIGURE_1],
    'controlled_source_scenarios': [f'm12_{level}' for level in CONTROLLED_FAMILIES],
    'tolerance_policy': 'all prefixes are shown; observed correlations and tolerance compliance are recorded without recalibration',
    'figure_2_interpretation': 'descriptive representative geometries; correlation is not isolated because anchors differ across levels',
    'figure_2_scenarios': [scenario for scenario, _ in FIGURE_2],
}
(OUT_DIR / 'true_pareto_comparison_metadata.json').write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8'
)

all_artifacts = controlled_artifacts + [png_2, pdf_2]
for path in all_artifacts:
    assert path.exists() and path.stat().st_size > 0
print(manifest.to_string(index=False))
print('Arquivos gerados:')
for path in all_artifacts:
    print(' -', path.relative_to(ROOT).as_posix())

## Leitura das figuras

- Em cada figura de família, as funções $f_1,\ldots,f_4$ são idênticas nos três painéis; $m=6$ acrescenta apenas $f_5$ e $f_6$, enquanto $m=12$ acrescenta as seis funções restantes. Essa construção complementar permite interpretar, dentro da família, as mudanças como efeito da inclusão de objetivos.
- A família baixa é mantida mesmo quando seus prefixos não satisfazem a tolerância de correlação original. Os valores observados e a coluna `within_original_tolerance` do manifesto tornam esse desvio explícito.
- Na Figura 2, o número de objetivos permanece igual a 12, mas as âncoras foram recalibradas entre os níveis. A figura deve ser interpretada apenas como comparação de geometrias representativas associadas a correlação baixa, média e alta, e não como identificação isolada do efeito da correlação.
- Os eixos mostram diretamente $f_1$ e $f_2$ na escala original das respostas. A subamostragem altera somente a densidade visual, não o conjunto de referência usado nas métricas.